# Nível 1 das Análises de Dados: Perfil e Distribuição dos dados.

# Bloco 1: Inventário e validação estrutural.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Definindo os caminhos das bases:

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_FINAL = ROOT / "data" / "final"

ARQUIVOS = {
    "vocabulario": DATA_FINAL / "dataset_vocabulario.csv",
    "tde": DATA_FINAL / "dataset_tde.csv",
    "alunos": DATA_FINAL / "dim_aluno.csv",
    "pareado_vocabulario": DATA_FINAL /  "pareado_vocabulario.csv",
    "pareado_tde": DATA_FINAL / "pareado_tde.csv"
}

for nome, caminho in ARQUIVOS.items():
    print(f"{nome}: {'Encontrado.' if caminho.exists() else "Não encontrado."}")
    print(f"{caminho}")

In [ ]:
# Carregando as bases:
bases = {}

for nome, caminho in ARQUIVOS.items():
    if not caminho.exists():
        raise FileNotFoundError(
            f"Base não encontrada: {caminho}"
        )
    bases[nome] = pd.read_csv(caminho, low_memory=False)

    print(
        f"{nome}: "
        f"{bases[nome].shape[0]:,} linhas x "
        f"{bases[nome].shape[1]:,} colunas"
    )

In [ ]:
# Construindo um inventário automático: Agora vamos produzir uma tabela com a estrutura das bases.

inventario = []

for nome, df in bases.items():
    inventario.append({
        "base": nome,
        "linhas": len(df),
        "colunas": len(df.columns),
        "colunas_numericas": len(df.select_dtypes(include='number').columns),
        "colunas_texto": len(df.select_dtypes(include=["object", "string"]).columns),
        "celulas_ausentes": int(df.isna().sum().sum()),
        "percentual_celulas_ausentes": round(df.isna().sum().sum() / df.size * 100, 2) if df.size else 0,
    })

df_inventario = pd.DataFrame(inventario)
df_inventario

In [ ]:
# Inspecionando as colunas:

for nome, df in bases.items():
    print(f"\n{'=' * 60}")
    print(f"BASE: {nome}")
    print(f"{'=' * 60}")

    print("\nColunas:")
    print(df.columns.tolist())

    print("\nTipos de dados:")
    display(df.dtypes.to_frame("tipo").T)

In [ ]:
# Virificando as chaves e duplicatas: Antes de contar alunos, precisamos investigar se aluno_id existe, está preenchido e se aparece repetido.

for nome in ["vocabulario", "tde", "alunos"]:
    df = bases[nome]

    print(f"\n--- {nome} ---")

    if "aluno_id" not in df.columns:
        print("ATENÇÃO: coluna aluno_id não encontrada.")
        continue

    print("IDs ausentes:", df["aluno_id"].isna().sum())
    print("IDs únicos:", df["aluno_id"].nunique())
    print("Linhas com ID repetido:", df["aluno_id"].duplicated().sum())

# Bloco 2: População e Cobertura.

* Quantos alunos únicos temos?
A primeira regra do projeto é: não somar as quantidades de alunos de Vocabulário e TDE para calcular o total geral. Um mesmo aluno pode estar presente nas duas bases.

In [ ]:
df_vocab = bases["vocabulario"]
df_tde = bases["tde"]
df_alunos = bases["alunos"]

ids_vocab = set(df_vocab["aluno_id"].dropna())
ids_tde = set(df_tde["aluno_id"].dropna())

ids_unicos_gerais = ids_vocab | ids_tde
ids_em_ambos = ids_vocab & ids_tde

kpis_populacao = pd.DataFrame([
    {
        "indicador": "Alunos únicos em vocabulário",
        "valor": len(ids_vocab)
    },

    {
        "indicador": "Alunos únicos em TDE",
        "valor": len(ids_tde)
    },

    {
        "indicador": "Alunos presentes nos dois instrumentos",
        "valor": len(ids_em_ambos)
    },

    {
        "indicador": "Alunos únicos no conjunto das avaliações",
        "valor": len(ids_unicos_gerais)
    },

    {
        "indicador": "Alunos na dimensão socioeconômica",
        "valor": df_alunos["aluno_id"].nunique()
    }
])

kpis_populacao

* Quantas avaliações existem por fase e instrumento?

In [ ]:
for nome in ["vocabulario", "tde"]:
    df = bases[nome]

    print(f"\nInstrumento: {nome}")

    if "fase" in df.columns:
        display(
            df.groupby("fase", dropna=False).agg(registros=("aluno_id", "size"), alunos_unicos=("aluno_id", "nunique")).reset_index().sort_values("fase")
        )
    else:
        print("Coluna fase não encontrada.")    

* Quantos registros existem por período?
O próximo passo é examinar a distribuição por período de avaliação.

In [ ]:
for nome in ["vocabulario", "tde"]:
    df = bases[nome]

    if {"fase", "periodo"}.issubset(df.columns):
        print(f"\n{nome}: fase × período")

        display(
            df.groupby(["fase", "periodo"], dropna=False)
              .agg(
                  registros=("aluno_id", "size"),
                  alunos_unicos=("aluno_id", "nunique"),
              )
              .reset_index()
              .sort_values(["fase", "periodo"])
        )

# Bloco 3: Distribuição escolar.
Vamos responder: em quais escolas, séries e turmas os dados estão concentrados, e como essa cobertura varia entre fases e instrumentos?

In [ ]:
colunas_escolares = ["escola", "serie", "turma", "fase", "periodo"]

for nome in ["vocabulario", "tde"]:
    df = bases[nome]

    print(f"\n===== {nome.upper()} =====")

    for coluna in colunas_escolares:
        if coluna in df.columns:
            print(f"\n{coluna}:")
            print("Categorias distintas:", df[coluna].nunique(dropna=True))
            print("Valores ausentes:", df[coluna].isna().sum())
            print("Exemplos:", df[coluna].dropna().unique()[:10])

* Criando uma função de distribuição.

In [ ]:
def resumir_distribuicao(df, dimensao, incluir_fase=True):
    colunas = [dimensao]

    if incluir_fase and "fase" in df.columns:
        colunas.insert(0, "fase")

    resumo = (
        df.groupby(colunas, dropna=False)
          .agg(
              registros=("aluno_id", "size"),
              alunos_unicos=("aluno_id", "nunique")
          )
          .reset_index()
    )

    resumo["percentual_registros"] = (
        resumo["registros"] / resumo["registros"].sum() * 100
    ).round(2)

    return resumo.sort_values(
        "registros", ascending=False
    )

* Gerando as tabelas de escolas, séries e turmas:

In [ ]:
distribuicoes_escolares = {}

for nome in ["vocabulario", "tde"]:
    df = bases[nome]

    for dimensao in ["escola", "serie", "turma"]:
        if dimensao in df.columns:
            chave = f"{nome}_{dimensao}"

            distribuicoes_escolares[chave] = (
                resumir_distribuicao(df, dimensao)
            )

            print(f"\n===== {chave.upper()} =====")
            display(distribuicoes_escolares[chave].head(20))

# Bloco 4: Completude das avaliações.

Aqui vamos responder: quanto dos instrumentos foi efetivamente respondido e quantas avaliações possuem dados suficientes para as análises posteriores?

Uma distinção importante: um valor ausente não é a mesma coisa que uma resposta incorreta. No Vocabulário, esperamos respostas válidas 0 e 1; no TDE, 0, 1 e 2. Valores fora dessas escalas precisam ser investigados.

* Identifique as colunas de itens

In [ ]:
itens_vocab = [
    c for c in df_vocab.columns
    if c.startswith("Q") and c[1:].isdigit()
]

itens_tde = [
    c for c in df_tde.columns
    if c.startswith("P") and c[1:].isdigit()
]

print("Itens de Vocabulário:", len(itens_vocab), itens_vocab)
print("Itens de TDE:", len(itens_tde), itens_tde)

Calcule a completude diretamente pelas respostas.
Vamos calcular novamente a completude a partir dos itens, para comparar com as colunas n_validos, pct_completude e completo que já existem nas bases.

In [ ]:
def analisar_completude(df, colunas_itens, valores_validos, instrumento):
    if not colunas_itens:
        raise ValueError(f"Nenhum item encontrado para {instrumento}.")

    respostas = df[colunas_itens].apply(
        pd.to_numeric, errors="coerce"
    )

    validas = respostas.isin(valores_validos)
    quantidade_validas = validas.sum(axis=1)
    quantidade_itens = len(colunas_itens)

    resultado = df[
        [c for c in ["aluno_id", "fase", "periodo", "escola", "serie", "turma"]
         if c in df.columns]
    ].copy()

    resultado["n_itens_analisados"] = quantidade_itens
    resultado["n_respostas_validas_calc"] = quantidade_validas
    resultado["n_respostas_ausentes_ou_invalidas"] = (
        quantidade_itens - quantidade_validas
    )
    resultado["pct_completude_calc"] = (
        quantidade_validas / quantidade_itens * 100
    ).round(2)
    resultado["avaliacao_completa_calc"] = (
        quantidade_validas == quantidade_itens
    )

    # Verifica valores numéricos fora da escala permitida.
    resultado["n_valores_fora_escala"] = (
        respostas.notna() & ~respostas.isin(valores_validos)
    ).sum(axis=1)

    resultado["instrumento"] = instrumento

    return resultado


completude_vocab = analisar_completude(
    df_vocab, itens_vocab, [0, 1], "Vocabulário"
)

completude_tde = analisar_completude(
    df_tde, itens_tde, [0, 1, 2], "TDE"
)

* Produzindo os indicadores gerais:

In [ ]:
def resumir_completude(df, nome):
    return {
        "instrumento": nome,
        "registros": len(df),
        "avaliacoes_completas": int(
            df["avaliacao_completa_calc"].sum()
        ),
        "percentual_avaliacoes_completas": round(
            df["avaliacao_completa_calc"].mean() * 100, 2
        ),
        "media_pct_completude": round(
            df["pct_completude_calc"].mean(), 2
        ),
        "respostas_ausentes_ou_invalidas": int(
            df["n_respostas_ausentes_ou_invalidas"].sum()
        ),
        "valores_fora_escala": int(
            df["n_valores_fora_escala"].sum()
        ),
    }


resumo_completude = pd.DataFrame([
    resumir_completude(completude_vocab, "Vocabulário"),
    resumir_completude(completude_tde, "TDE"),
])

display(resumo_completude)

Compare por fase e período: 

In [ ]:
for nome, df in [
    ("Vocabulário", completude_vocab),
    ("TDE", completude_tde),
]:
    print(f"\n===== COMPLETUDE: {nome} =====")

    colunas_grupo = [
        c for c in ["fase", "periodo"] if c in df.columns
    ]

    display(
        df.groupby(colunas_grupo, dropna=False)
          .agg(
              registros=("aluno_id", "size"),
              completas=("avaliacao_completa_calc", "sum"),
              completude_media=("pct_completude_calc", "mean"),
              valores_fora_escala=("n_valores_fora_escala", "sum"),
          )
          .round(2)
          .reset_index()
    )

Também vale verificar se os cálculos correspondem às métricas do pipeline:

In [ ]:
for nome, original, calculada in [
    ("Vocabulário", df_vocab, completude_vocab),
    ("TDE", df_tde, completude_tde),
]:
    print(f"\n===== COMPARAÇÃO: {nome} =====")

    if "n_validos" in original.columns:
        display(pd.DataFrame({
            "n_validos_pipeline": original["n_validos"].describe(),
            "n_validos_calculado": (
                calculada["n_respostas_validas_calc"].describe()
            ),
        }))

    if "pct_completude" in original.columns:
        display(pd.DataFrame({
            "pct_completude_pipeline": original["pct_completude"].describe(),
            "pct_completude_calculada": (
                calculada["pct_completude_calc"].describe()
            ),
        }))

# Bloco 5: Distribuição socioeconômica.

Agora vamos descrever os atributos disponíveis em dim_aluno.csv. Isso será importante para entender a cobertura das informações e, mais adiante, estudar associações entre contexto socioeconômico e desempenho.

As colunas identificadas são sexo, faixa_etaria, cor_raca, zona, necessidade_especial e fonte_socio.

Calcule a cobertura de cada atributo:

In [ ]:
atributos_socio = [
    "sexo",
    "faixa_etaria",
    "cor_raca",
    "zona",
    "necessidade_especial",
    "fonte_socio",
]

resumo_socio = []

for coluna in atributos_socio:
    if coluna not in df_alunos.columns:
        continue

    valores = df_alunos[coluna]
    preenchidos = valores.notna() & valores.astype(str).str.strip().ne("")

    resumo_socio.append({
        "atributo": coluna,
        "alunos_na_dimensao": len(df_alunos),
        "valores_preenchidos": int(preenchidos.sum()),
        "valores_ausentes_ou_vazios": int((~preenchidos).sum()),
        "percentual_preenchido": round(
            preenchidos.mean() * 100, 2
        ),
        "categorias_distintas": valores[preenchidos].nunique(),
    })

df_resumo_socio = pd.DataFrame(resumo_socio)
display(df_resumo_socio)

Distribuição de cada atributo:

In [ ]:
distribuicoes_socio = {}

for coluna in atributos_socio:
    if coluna not in df_alunos.columns:
        continue

    valores = (
        df_alunos[coluna]
        .fillna("Ausente")
        .astype(str)
        .str.strip()
        .replace("", "Ausente")
    )

    resumo = (
        valores.value_counts(dropna=False)
        .rename_axis("categoria")
        .reset_index(name="alunos")
    )

    resumo["percentual"] = (
        resumo["alunos"] / len(df_alunos) * 100
    ).round(2)

    distribuicoes_socio[coluna] = resumo

    print(f"\n===== {coluna.upper()} =====")
    display(resumo)

Verifique a cobertura socioeconômica por instrumento:

In [ ]:
ids_socio = set(df_alunos["aluno_id"].dropna())

for nome in ["vocabulario", "tde"]:
    df = bases[nome]
    ids_avaliados = set(df["aluno_id"].dropna())

    ids_com_socio = ids_avaliados & ids_socio
    ids_sem_socio = ids_avaliados - ids_socio

    print(f"\n===== COBERTURA SOCIOECONÔMICA: {nome.upper()} =====")
    print("Alunos únicos no instrumento:", len(ids_avaliados))
    print("Alunos com registro na dimensão:", len(ids_com_socio))
    print("Alunos sem registro na dimensão:", len(ids_sem_socio))
    print(
        "Cobertura (%):",
        round(len(ids_com_socio) / len(ids_avaliados) * 100, 2)
        if ids_avaliados else 0
    )

# Bloco 6: Cobertura longitudinal.

Queremos responder: quantos alunos aparecem em mais de uma fase, quantos possuem registros nos dois instrumentos e quão ampla é a cobertura das avaliações pré/pós?

Precisamos separar três conceitos:
- Presença em várias fases: o identificador do aluno aparece em mais de uma fase.
- Presença nos dois instrumentos: o aluno aparece em Vocabulário e TDE.
- Avaliação pareada: existe um registro que associa pré e pós segundo as regras do pipeline.
Esses conceitos não são equivalentes.

Quantos alunos aparecem em cada fase?

In [ ]:
def resumir_presenca_fases(df, instrumento):
    resumo = (
        df.groupby("fase", dropna=False)
          .agg(
              alunos_unicos=("aluno_id", "nunique"),
              registros=("aluno_id", "size"),
          )
          .reset_index()
    )

    resumo.insert(0, "instrumento", instrumento)
    return resumo


presenca_fases = pd.concat([
    resumir_presenca_fases(df_vocab, "Vocabulário"),
    resumir_presenca_fases(df_tde, "TDE"),
], ignore_index=True)

display(presenca_fases)

Quantos alunos aparecem em cada combinação de fases?

In [ ]:
def analisar_recorrencia_fases(df, instrumento):
    fases_por_aluno = (
        df.dropna(subset=["aluno_id", "fase"])
          .groupby("aluno_id")["fase"]
          .agg(lambda s: set(s.astype(str)))
    )

    distribuicao = (
        fases_por_aluno.map(len)
        .value_counts()
        .sort_index()
        .rename_axis("quantidade_fases")
        .reset_index(name="alunos")
    )

    distribuicao["instrumento"] = instrumento
    return distribuicao


recorrencia = pd.concat([
    analisar_recorrencia_fases(df_vocab, "Vocabulário"),
    analisar_recorrencia_fases(df_tde, "TDE"),
], ignore_index=True)

display(
    recorrencia[
        ["instrumento", "quantidade_fases", "alunos"]
    ]
)

A interpretação será:
- 1: alunos identificados em apenas uma fase;
- 2: alunos identificados em duas fases;
- 3: alunos identificados nas três fases.

Calculando a interseção entre fases:

In [ ]:
from itertools import combinations

def intersecao_fases(df, instrumento):
    ids_por_fase = {
        str(fase): set(grupo["aluno_id"].dropna())
        for fase, grupo in df.groupby("fase", dropna=False)
        if pd.notna(fase)
    }

    resultados = []

    for fase_a, fase_b in combinations(ids_por_fase.keys(), 2):
        ids_a = ids_por_fase[fase_a]
        ids_b = ids_por_fase[fase_b]
        intersecao = ids_a & ids_b

        resultados.append({
            "instrumento": instrumento,
            "fase_a": fase_a,
            "fase_b": fase_b,
            "alunos_fase_a": len(ids_a),
            "alunos_fase_b": len(ids_b),
            "alunos_presentes_em_ambas": len(intersecao),
            "percentual_fase_a_presente_em_b": round(
                len(intersecao) / len(ids_a) * 100, 2
            ) if ids_a else 0,
            "percentual_fase_b_presente_em_a": round(
                len(intersecao) / len(ids_b) * 100, 2
            ) if ids_b else 0,
        })

    return pd.DataFrame(resultados)


intersecoes = pd.concat([
    intersecao_fases(df_vocab, "Vocabulário"),
    intersecao_fases(df_tde, "TDE"),
], ignore_index=True)

display(intersecoes)

Comparando os instrumentos dentro de cada fase:

In [ ]:
fases_vocab = {
    str(fase): set(grupo["aluno_id"].dropna())
    for fase, grupo in df_vocab.groupby("fase")
}

fases_tde = {
    str(fase): set(grupo["aluno_id"].dropna())
    for fase, grupo in df_tde.groupby("fase")
}

comparacao_instrumentos = []

for fase in sorted(set(fases_vocab) & set(fases_tde)):
    ids_v = fases_vocab[fase]
    ids_t = fases_tde[fase]
    comuns = ids_v & ids_t

    comparacao_instrumentos.append({
        "fase": fase,
        "alunos_vocabulario": len(ids_v),
        "alunos_tde": len(ids_t),
        "alunos_nos_dois_instrumentos": len(comuns),
        "percentual_vocab_com_tde": round(
            len(comuns) / len(ids_v) * 100, 2
        ) if ids_v else 0,
        "percentual_tde_com_vocab": round(
            len(comuns) / len(ids_t) * 100, 2
        ) if ids_t else 0,
    })

df_comparacao_instrumentos = pd.DataFrame(comparacao_instrumentos)
display(df_comparacao_instrumentos)

Inspecionando as bases pareadas. Como o pipeline já gerou bases pareadas, vamos verificar sua cobertura e a validade dos registros:

In [ ]:
def resumir_pareado(df, instrumento):
    resumo = {
        "instrumento": instrumento,
        "registros_pareados": len(df),
        "alunos_unicos": df["aluno_id"].nunique(),
    }

    for coluna in ["completo_pre", "completo_pos"]:
        if coluna in df.columns:
            resumo[f"{coluna}_verdadeiros"] = int(
                df[coluna].astype(str).str.lower()
                .isin(["true", "1", "sim"]).sum()
            )

    if {"total_pre", "total_pos"}.issubset(df.columns):
        pre = pd.to_numeric(df["total_pre"], errors="coerce")
        pos = pd.to_numeric(df["total_pos"], errors="coerce")

        resumo["pre_ausente"] = int(pre.isna().sum())
        resumo["pos_ausente"] = int(pos.isna().sum())
        resumo["pre_e_pos_preenchidos"] = int(
            (pre.notna() & pos.notna()).sum()
        )

    return resumo


resumo_pareado = pd.DataFrame([
    resumir_pareado(bases["pareado_vocabulario"], "Vocabulário"),
    resumir_pareado(bases["pareado_tde"], "TDE"),
])

display(resumo_pareado)

In [ ]:

# Validação final do nível 1 — BASES PAREADAS

for nome, df in [
    ("Vocabulário", bases["pareado_vocabulario"]),
    ("TDE", bases["pareado_tde"]),
]:
    print(f"\n{'=' * 55}")
    print(f"VALIDAÇÃO: {nome}")
    print(f"{'=' * 55}")

    # 1. Identificação dos registros repetidos por aluno e fase
    chave = ["aluno_id", "fase"]
    repetidos = df.duplicated(subset=chave, keep=False)

    print(f"Registros: {len(df):,}")
    print(f"Alunos únicos: {df['aluno_id'].nunique():,}")
    print(f"Linhas em combinações aluno-fase repetidas: {repetidos.sum():,}")
    print(
        "Combinações aluno-fase únicas:",
        df[chave].drop_duplicates().shape[0]
    )

    # 2. Verificação dos totais e percentuais
    for coluna in ["total_pre", "total_pos", "pct_pre", "pct_pos"]:
        if coluna in df.columns:
            valores = pd.to_numeric(df[coluna], errors="coerce")
            print(
                f"{coluna}: ausentes={valores.isna().sum()}, "
                f"mínimo={valores.min()}, máximo={valores.max()}"
            )

    # 3. Comparação entre os totais preenchidos e as flags de completude
    for coluna in ["completo_pre", "completo_pos"]:
        if coluna in df.columns:
            print(f"\n{coluna}:")
            print(df[coluna].value_counts(dropna=False).to_string())

    # 4. Variação pré/pós apenas como diagnóstico inicial
    pre = pd.to_numeric(df["total_pre"], errors="coerce")
    pos = pd.to_numeric(df["total_pos"], errors="coerce")
    pares_preenchidos = pre.notna() & pos.notna()

    print(
        "\nRegistros com pré e pós preenchidos:",
        int(pares_preenchidos.sum())
    )
    print(
        "Diferença média pós - pré (diagnóstico, ainda sem validação):",
        round((pos[pares_preenchidos] - pre[pares_preenchidos]).mean(), 2)
        if pares_preenchidos.any() else "Sem pares válidos"
    )

In [ ]:
# Verificação de consistência e cobertura.

def auditar_instrumento(nome, chave, coluna_total, prefixo, escala):
    df = bases[chave].copy()
    itens = sorted(
        [c for c in df.columns
         if c.startswith(prefixo) and c[len(prefixo):].isdigit()],
        key=lambda c: int(c[len(prefixo):])
    )

    print(f"\n{'=' * 60}")
    print(f"1. CONSISTÊNCIA — {nome}")
    print(f"{'=' * 60}")

    if not itens or coluna_total not in df.columns:
        print("ERRO: não foi possível identificar os itens ou o total.")
        return

    respostas = df[itens].apply(pd.to_numeric, errors="coerce")
    total = pd.to_numeric(df[coluna_total], errors="coerce")

    valores_validos = [0, 1] if prefixo == "Q" else [0, 1, 2]
    respostas_validas = respostas.isin(valores_validos)
    completa = respostas_validas.all(axis=1)

    # Comparação do total apenas em avaliações integralmente completas.
    soma_itens = respostas.sum(axis=1, min_count=len(itens))
    divergencia_total = (
        completa
        & total.notna()
        & ~np.isclose(total, soma_itens, atol=1e-8, rtol=0)
    )

    fora_escala = (
        respostas.notna() & ~respostas.isin(valores_validos)
    ).sum().sum()

    print(f"Registros: {len(df):,}")
    print(f"Itens identificados: {len(itens)}")
    print(f"Avaliações completas pelos itens: {int(completa.sum()):,}")
    print(f"Totais ausentes: {int(total.isna().sum()):,}")
    print(f"Totais divergentes nas avaliações completas: "
          f"{int(divergencia_total.sum()):,}")
    print(f"Respostas numéricas fora da escala: {int(fora_escala):,}")
    print(f"Total mínimo/máximo observado: {total.min()} / {total.max()}")
    print(f"Limite teórico do instrumento: {escala}")

    if total.notna().any():
        fora_limite = ((total < 0) | (total > escala)).sum()
        print(f"Totais fora do limite teórico: {int(fora_limite):,}")


auditar_instrumento(
    "Vocabulário",
    "vocabulario",
    "total_vocabulario",
    "Q",
    50
)

auditar_instrumento(
    "TDE",
    "tde",
    "total_tde",
    "P",
    80
)


# 2. COBERTURA DA DIMENSÃO SOCIOECONÔMICA

print(f"\n{'=' * 60}")
print("2. COBERTURA SOCIOECONÔMICA")
print(f"{'=' * 60}")

ids_vocab = set(bases["vocabulario"]["aluno_id"].dropna().unique())
ids_tde = set(bases["tde"]["aluno_id"].dropna().unique())
ids_dim = set(bases["alunos"]["aluno_id"].dropna().unique())

ids_uniao = ids_vocab | ids_tde
ids_sem_dim = ids_uniao - ids_dim

print(f"Alunos únicos em Vocabulário: {len(ids_vocab):,}")
print(f"Alunos únicos em TDE: {len(ids_tde):,}")
print(f"Alunos únicos na união dos instrumentos: {len(ids_uniao):,}")
print(f"Alunos únicos em dim_aluno: {len(ids_dim):,}")
print(f"Alunos da união sem correspondência socioeconômica: "
      f"{len(ids_sem_dim):,}")
print(f"Cobertura socioeconômica da união: "
      f"{100 * len(ids_uniao & ids_dim) / len(ids_uniao):.2f}%")

for nome, ids in [("Vocabulário", ids_vocab), ("TDE", ids_tde)]:
    ausentes = ids - ids_dim
    print(
        f"{nome}: {len(ausentes):,} alunos sem dimensão "
        f"({100 * len(ausentes) / len(ids):.2f}%)"
    )

print(
    "IDs ausentes na dimensão, mas presentes nos instrumentos, "
    "serão mantidos nas análises de desempenho."
)


# 3. REGRAS E VERIFICAÇÕES DOS INDICADORES

print(f"\n{'=' * 60}")
print("3. REGRAS DOS INDICADORES")
print(f"{'=' * 60}")

print("KPI geral: contar a união de aluno_id entre instrumentos;")
print("não somar os alunos únicos de Vocabulário e TDE.")
print("KPI por categoria: contar aluno_id único dentro do instrumento.")
print("Avaliação completa: todos os itens devem estar na escala válida.")

for nome, chave in [
    ("Vocabulário", "pareado_vocabulario"),
    ("TDE", "pareado_tde")
]:
    df = bases[chave].copy()
    pre = pd.to_numeric(df["total_pre"], errors="coerce")
    pos = pd.to_numeric(df["total_pos"], errors="coerce")

    flags_pre = (
        df["completo_pre"].astype(str).str.strip().str.lower()
        .isin(["true", "1", "sim"])
    )
    flags_pos = (
        df["completo_pos"].astype(str).str.strip().str.lower()
        .isin(["true", "1", "sim"])
    )

    pares_completos = (
        flags_pre & flags_pos & pre.notna() & pos.notna()
    )

    chave_repetida = df.duplicated(
        subset=["aluno_id", "fase"], keep=False
    ).sum()

    print(f"\n{nome}:")
    print(f"  Alunos únicos: {df['aluno_id'].nunique():,}")
    print(f"  Chaves aluno-fase repetidas: {int(chave_repetida):,}")
    print(f"  Pares pré/pós completos pelas flags: "
          f"{int(pares_completos.sum()):,}")
    print(f"  Pares excluídos por incompletude/total ausente: "
          f"{int((~pares_completos).sum()):,}")

print("\nAuditoria concluída. Nenhuma base foi modificada.")